# 20a_split_train_val_test — 훈련/검증/시험 데이터 나누기 (신규)

**한 줄 요약:** 최종 학습데이터를 **훈련(train)·검증(validation)·시험(test)** 세 덩어리로 나눈다. 각 덩어리의 active:inactive 비율은 그대로 유지(**층화, stratified**).
**왜 필요:** 지금까지는 교차검증만 했는데, 하이퍼파라미터를 **검증셋으로 고르고** 최종 성능은 **시험셋으로 딱 한 번** 재기 위함(공정한 평가).
**비율:** train 70% / val 15% / test 15%.
**출력:** 용량 절약 위해 `canonical_smiles → split` 표만 저장(모델 노트북이 이걸 원본에 붙여서 씀).

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기
분할 도구(`train_test_split`)를 가져온다.

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())
import pandas as pd
from sklearn.model_selection import train_test_split

🔎 **코드 뜯어보기 (준비)**
- `from sklearn.model_selection import train_test_split` : 데이터를 두 덩어리로 **무작위(층화 가능) 분할**하는 함수.

### 셀 1 — 층화 분할 (train/val/test)
test를 먼저 떼고, 남은 것에서 val을 떼어 train/val/test 세 덩어리로. 각 덩어리 potency 비율은 유지.

In [ ]:
# 최종 학습데이터를 train/validation/test 로 '층화(stratified)' 분할
SRC = "data/HSD17B13_final_training_1to1_v2.csv"
OUT = "data/HSD17B13_split_1to1.csv"      # canonical_smiles -> split(train/val/test) 만 저장(용량 절약)
TEST = 0.15                                # 시험 15%
VAL = 0.15                                 # 검증 15% (나머지 70% 훈련)

# 무거운 지문 열은 안 읽고, 라벨만 필요 (분할은 potency 비율만 맞추면 됨)
df = pd.read_csv(SRC, usecols=["canonical_smiles", "potency"])
y = df["potency"]

# 1) 먼저 test 15% 떼기 (potency 비율 유지)
tr_val, te = train_test_split(df, test_size=TEST, stratify=y, random_state=42)
# 2) 남은 85%에서 val 을 떼기 (전체의 15%가 되도록 비율 환산)
val_ratio = VAL / (1 - TEST)
tr, va = train_test_split(tr_val, test_size=val_ratio, stratify=tr_val["potency"], random_state=42)

split = pd.concat([
    tr.assign(split="train"), va.assign(split="val"), te.assign(split="test"),
])[["canonical_smiles", "split"]]
split.to_csv(OUT, index=False)

print("분할 결과(개수):")
for name, part in [("train", tr), ("val", va), ("test", te)]:
    vc = part["potency"].value_counts().to_dict()
    print(f"  {name:5s}: {len(part):5d}  (active {vc.get(1,0)}, inactive {vc.get(0,0)})")
print("저장:", OUT)

🔎 **코드 뜯어보기 (셀 1)**
- `pd.read_csv(SRC, usecols=["canonical_smiles","potency"])` : 분할엔 라벨만 있으면 되니 **두 열만** 읽음(5천 열 다 안 읽어 빠름).
- `train_test_split(df, test_size=0.15, stratify=y, random_state=42)` : 15%를 test로 분리. **stratify=y**=active:inactive 비율을 두 쪽 모두 유지. `random_state`=재현용 씨앗.
- `val_ratio = VAL/(1-TEST)` : 남은 85% 중에서 몇 %를 떼야 **전체의 15%**가 되는지 환산.
- `.assign(split="train")` : 각 덩어리에 소속 라벨 열 추가 후 `pd.concat`으로 합쳐 저장.